# 🌍 Search & Load CMIP6 Data via ESGF / OPeNDAP

This notebook searches for and loads CMIP6 climate model output directly over the network using
[**Earth System Grid Federation**](https://esgf.llnl.gov/) (ESGF) infrastructure — the backbone of the CMIP6 distribution system.

Two technologies do the heavy lifting:

| Technology | Role |
|---|---|
| [**ESGF Search API**](https://github.com/ESGF/esgf.github.io/wiki/ESGF_Search_REST_API) | Discover *which* files hold the data we want |
| [**OPeNDAP**](https://www.opendap.org/) | Stream those files lazily over HTTP — no full download |

**What we'll do**

1. Query ESGF for near-surface air temperature (`tas`) from the NCAR **CESM2** historical run
2. Open the remote files lazily with `xarray` + `dask`
3. Map the temperature field on a proper geographic projection
4. Build an **area-weighted global-mean** temperature series
5. Compute the **warming anomaly** relative to an 1850–1900 pre-industrial baseline

> ⚠️ **Live data.** Cells pull from ESGF nodes (LLNL / UCAR / DIAS) in real time. If a search or load cell errors, it is almost always a node being temporarily down or relocated — re-run, or swap the search node.

## ⚙️ Setup

Imports plus a little plot styling so the figures look clean.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature

xr.set_options(display_style='html')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# --- light, consistent plot styling ---
mpl.rcParams.update({
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'font.size': 11,
})

## 🔎 Search using the ESGF API

The helper below wraps the ESGF Solr search endpoint. Give it CMIP6 facets as keyword arguments
(`variable_id`, `experiment_id`, `source_id`, …) and it returns the matching OPeNDAP URLs.

*Adapted from an ESGF reference document; see the [REST API docs](https://github.com/ESGF/esgf.github.io/wiki/ESGF_Search_REST_API#results-pagination).*

In [ ]:
import requests

def esgf_search(server="https://esgf-node.llnl.gov/esg-search/search",
                files_type="OPENDAP", local_node=True, project="CMIP6",
                verbose=False, format="application%2Fsolr%2Bjson",
                use_csrf=False, **search):
    """Query the ESGF search API and return a sorted list of file URLs."""
    client = requests.session()
    payload = search
    payload["project"] = project
    payload["type"] = "File"
    if local_node:
        payload["distrib"] = "false"
    if use_csrf:
        client.get(server)
        if 'csrftoken' in client.cookies:
            csrftoken = client.cookies['csrftoken']   # Django 1.6+
        else:
            csrftoken = client.cookies['csrf']         # older
        payload["csrfmiddlewaretoken"] = csrftoken

    payload["format"] = format

    offset = 0
    numFound = 10000
    all_files = []
    files_type = files_type.upper()
    while offset < numFound:
        payload["offset"] = offset
        url_keys = ["{}={}".format(k, payload[k]) for k in payload]
        url = "{}/?{}".format(server, "&".join(url_keys))
        if verbose:
            print(url)
        r = client.get(url)
        r.raise_for_status()
        resp = r.json()["response"]
        numFound = int(resp["numFound"])
        resp = resp["docs"]
        offset += len(resp)
        for d in resp:
            for f in d["url"]:
                sp = f.split("|")
                if sp[-1] == files_type:
                    all_files.append(sp[0].split(".html")[0])
    return sorted(all_files)

Search for monthly near-surface air temperature (`tas`) from **CESM2**, historical experiment, member `r10i1p1f1`:

In [ ]:
result = esgf_search(activity_id='CMIP', table_id='Amon', variable_id='tas',
                     experiment_id='historical', institution_id='NCAR',
                     source_id='CESM2', member_id='r10i1p1f1')

print(f'Found {len(result)} file URLs across mirror nodes.\n')
for u in result:
    print(' •', u)

## 📦 Load data with Xarray

These are OPeNDAP endpoints. `xarray` + `netCDF4` open them **lazily** — only metadata is read now;
the actual arrays stream on demand as `dask` chunks.

The same data is mirrored on several nodes, so we keep just one node's set of files (the last four).

In [ ]:
files_to_open = result[-4:]

ds = xr.open_mfdataset(files_to_open, combine='by_coords')
ds

## 🗺️ Map a single month

Instead of a bare lat/lon grid, we render the field on a **Robinson projection** with coastlines
and a labelled colorbar.

In [ ]:
snapshot = ds.tas.sel(time='1950-01').squeeze()

fig = plt.figure(figsize=(11, 5.5))
ax = plt.axes(projection=ccrs.Robinson())

p = snapshot.plot(
    ax=ax, transform=ccrs.PlateCarree(),
    cmap='RdBu_r', robust=True,
    cbar_kwargs={'label': 'Near-surface air temperature (K)', 'shrink': 0.7, 'pad': 0.04},
)
ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
ax.gridlines(alpha=0.3)
ax.set_global()
ax.set_title('CESM2 historical — surface air temperature, Jan 1950', fontweight='bold')
plt.show()

### Seasonal climatology

A four-panel seasonal mean over the full 1850–2014 record. This triggers a real computation across
all timesteps streamed over OPeNDAP, so give it a moment.

In [ ]:
seasonal = ds.tas.groupby('time.season').mean('time')
seasons = ['DJF', 'MAM', 'JJA', 'SON']

fig, axes = plt.subplots(2, 2, figsize=(12, 7),
                         subplot_kw={'projection': ccrs.Robinson()})
for ax, season in zip(axes.flat, seasons):
    seasonal.sel(season=season).plot(
        ax=ax, transform=ccrs.PlateCarree(),
        cmap='RdBu_r', robust=True, add_colorbar=False,
    )
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.set_global()
    ax.set_title(season)

fig.suptitle('Seasonal mean surface air temperature (1850–2014)',
             fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## 🌡️ Area-weighted global-mean temperature

A simple average over grid cells is biased — cells near the poles are much smaller than at the equator.
We weight by each cell's area (`areacella`) to get a true global mean.

In [ ]:
files_area = esgf_search(variable_id='areacella', activity_id='CMIP',
                         experiment_id='historical', institution_id='NCAR',
                         source_id='CESM2')
ds_area = xr.open_dataset(files_area[0])
ds_area

In [ ]:
total_area = ds_area.areacella.sum(dim=['lon', 'lat'])
ta_timeseries = (ds.tas * ds_area.areacella).sum(dim=['lon', 'lat']) / total_area
ta_timeseries

Data loads lazily as dask arrays; here we trigger the computation explicitly.

In [ ]:
%time ta_timeseries = ta_timeseries.load()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))

ta_timeseries.plot(ax=ax, color='0.7', lw=0.8, label='Monthly')
ta_timeseries.rolling(time=12, center=True).mean().plot(
    ax=ax, color='crimson', lw=2, label='12-month rolling mean')

ax.set_title('Global Mean Surface Air Temperature — CESM2 historical')
ax.set_ylabel('Temperature (K)')
ax.set_xlabel('')
ax.legend(frameon=False)
plt.show()

## 📈 Warming relative to pre-industrial

Climate signals read more clearly as **anomalies**. We take an 1850–1900 baseline, subtract it,
and average to annual values — warm years above the line, cool years below.

In [ ]:
baseline = ta_timeseries.sel(time=slice('1850', '1900')).mean()
anom = ta_timeseries - baseline
annual = anom.groupby('time.year').mean()

fig, ax = plt.subplots(figsize=(11, 4.5))
years = annual['year'].values
vals = annual.values

ax.fill_between(years, vals, where=(vals >= 0), color='#d73027', alpha=0.8, interpolate=True)
ax.fill_between(years, vals, where=(vals < 0),  color='#4575b4', alpha=0.8, interpolate=True)
ax.axhline(0, color='k', lw=0.8)

ax.set_title('Annual global-mean temperature anomaly\n(relative to 1850–1900 baseline)')
ax.set_ylabel('Anomaly (K)')
ax.set_xlabel('Year')
ax.margins(x=0.01)
plt.show()

print(f'Total warming (1850–1900 → 2010–2014): '
      f'{float(annual.sel(year=slice(2010, 2014)).mean()):.2f} K')

## 💾 Data footprint

How large is the full dataset, in MB per simulation year?

In [ ]:
ds.nbytes / 1e6 / (ds.sizes['time'] / 12)

---
## Summary

We searched ESGF for CMIP6 `tas` output, streamed it over OPeNDAP without downloading whole files,
mapped it on geographic projections, and reduced it to an area-weighted global-mean warming signal.

**Try next**

- Swap `source_id='CESM2'` for another model (e.g. `'GFDL-ESM4'`, `'MPI-ESM1-2-LR'`)
- Change `experiment_id` to `'ssp585'` (with `activity_id='ScenarioMIP'`) to project future warming
- Add ensemble members and plot their spread